In [2]:
# @title 🚑 RELAY — Bridging the Gap Before Arrival (Full Bilingual & Live Voice Edition)
!pip install -q google-genai pydantic requests

import os, re, csv, json, time, uuid, base64, socket, threading, getpass
from datetime import datetime
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse
from typing import Optional, List, Dict, Any
from pydantic import BaseModel, Field
from google import genai
from google.genai import types
import requests

# 1. إعداد المفاتيح البرمجية بأمان (Colab Secrets أو إدخال مباشر)
def get_secret_or_env(key: str, prompt_if_missing: bool = False) -> str:
    val = os.environ.get(key, "").strip()
    if not val:
        try:
            from google.colab import userdata
            val = (userdata.get(key) or "").strip()
        except Exception:
            val = ""
    if not val and prompt_if_missing:
        val = getpass.getpass(f"🔑 Enter your {key}: ").strip()
    if val:
        os.environ[key] = val
    return val

GEMINI_API_KEY = get_secret_or_env("GEMINI_API_KEY", prompt_if_missing=True)
TELEGRAM_BOT_TOKEN = get_secret_or_env("TELEGRAM_BOT_TOKEN", prompt_if_missing=False)
TELEGRAM_CHAT_ID = get_secret_or_env("TELEGRAM_CHAT_ID", prompt_if_missing=False)

client = genai.Client(api_key=GEMINI_API_KEY)
MODELS_PRIORITY = ["gemini-2.5-flash", "gemini-2.5-flash-lite", "gemini-flash-lite-latest"]
LOG_FILE = "relay_handover_audit_log.csv"
CASES_STORE: List[Dict[str, Any]] = []
STORE_LOCK = threading.Lock()

# 2. حماية البيانات الصحية المحمية (PHI Redaction & Masking)
def redact_phi_for_llm(text: str) -> str:
    if not text:
        return ""
    cleaned = re.sub(r"\b\d{14}\b", "[NATIONAL_ID_REDACTED]", text)
    cleaned = re.sub(r"(?:\+?20|0)?1[0125]\d{8}\b", "[PHONE_REDACTED]", cleaned)
    cleaned = re.sub(
        r"(?i)(patient\s+name\s+is|name\s+is|المريض|اسمه|اسمها)\s+([A-Za-z\u0600-\u06FF]+(?:\s+[A-Za-z\u0600-\u06FF]+){0,2})",
        r"\1 [PATIENT_NAME_REDACTED]",
        cleaned,
    )
    return cleaned

def mask_national_id(nid: str) -> str:
    digits = re.sub(r"\D", "", str(nid or ""))
    if len(digits) >= 4:
        return "*" * max(10, len(digits) - 4) + digits[-4:]
    return "**************" if digits else "N/A"

def generate_patient_id() -> str:
    date_part = datetime.now().strftime("%Y%m%d")
    with STORE_LOCK:
        seq = len(CASES_STORE) + 1
    suffix = uuid.uuid4().hex[:3].upper()
    return f"PT-{date_part}-{seq:03d}-{suffix}"

# 3. نماذج البيانات المهيكلة (Pydantic Schemas)
class ExtractedCaseFields(BaseModel):
    patient_name: Optional[str] = Field(default=None, description="Full patient name if stated")
    age: Optional[int] = Field(default=None, description="Patient age in years (0 for infants)")
    gender: Optional[str] = Field(default=None, description="Must be 'Male' or 'Female' if mentioned, else null")
    national_id: Optional[str] = Field(default=None, description="14-digit national ID without spaces")
    sbp: Optional[int] = Field(default=None, description="Systolic Blood Pressure in mmHg")
    dbp: Optional[int] = Field(default=None, description="Diastolic Blood Pressure in mmHg")
    hr: Optional[int] = Field(default=None, description="Heart Rate in bpm")
    spo2: Optional[int] = Field(default=None, description="Oxygen saturation SpO2 percentage")
    rr: Optional[int] = Field(default=None, description="Respiratory rate breaths/min")
    temp: Optional[float] = Field(default=None, description="Body temperature in Celsius")
    avpu: Optional[str] = Field(default=None, description="One of: A, V, P, U")
    gcs: Optional[int] = Field(default=None, description="Glasgow Coma Scale score 3-15")
    medications: Optional[str] = Field(default=None, description="Comma-separated medications or drug classes, e.g. 'Plavix, Unknown blood thinners'")
    eta: Optional[int] = Field(default=None, description="Estimated Time of Arrival in minutes")

class ClinicalDraftResponse(BaseModel):
    clinical_summary: str
    suggested_department: str
    immediate_actions: List[str]
    staff_to_alert: List[str]
    equipment: List[str]
    labs_imaging: List[str]
    medications_prep: List[str]
    precautions: List[str]

# 4. محرك القواعد الفسيولوجية الحتمي (Deterministic Physiological Rules Engine)
def assess_vitals(data: Dict[str, Any], lang: str = "en") -> Dict[str, Any]:
    is_en = (lang == "en")
    sbp = data.get("sbp")
    dbp = data.get("dbp")
    hr = data.get("hr")
    spo2 = data.get("spo2")
    rr = data.get("rr")
    temp = data.get("temp")
    avpu = (data.get("avpu") or "").upper()
    gcs = data.get("gcs")
    age = data.get("age")

    map_val = None
    if sbp is not None and dbp is not None and sbp > 0 and dbp > 0:
        map_val = round(dbp + (sbp - dbp) / 3.0)

    si_val = None
    if hr is not None and sbp is not None and sbp > 0:
        si_val = round(hr / float(sbp), 2)

    alerts = []
    min_score = 2
    tier = "GREEN"

    def escalate(new_tier: str, score_floor: int):
        nonlocal tier, min_score
        order = {"GREEN": 1, "ORANGE": 2, "RED": 3}
        if order[new_tier] > order[tier]:
            tier = new_tier
        if score_floor > min_score:
            min_score = score_floor

    if spo2 is not None:
        if spo2 < 90:
            escalate("RED", 9)
            alerts.append(f"SpO2 {spo2}% — Severe hypoxia (<90%)" if is_en else f"نقص أكسجة حاد SpO2 {spo2}% (<90%)")
        elif spo2 <= 94:
            escalate("ORANGE", 6)
            alerts.append(f"SpO2 {spo2}% — Borderline hypoxia (90–94%)" if is_en else f"نقص أكسجة متوسط SpO2 {spo2}% (90–94%)")

    if sbp is not None:
        if sbp < 90:
            escalate("RED", 9)
            alerts.append(f"SBP {sbp} mmHg — Hypotension / Shock threshold" if is_en else f"الضغط الانقباضي {sbp} — هبوط ضغط / صدمة")
        elif sbp <= 100 or sbp >= 180:
            escalate("ORANGE", 6)
            alerts.append(f"SBP {sbp} mmHg — Hemodynamic instability" if is_en else f"الضغط الانقباضي {sbp} — عدم استقرار ديناميكي")

    if map_val is not None and map_val < 65:
        escalate("RED", 9)
        alerts.append(f"Estimated MAP {map_val} mmHg — Low organ perfusion (<65)" if is_en else f"متوسط الضغط الشرياني التقديري MAP={map_val} mmHg — انخفاض التروية")

    if hr is not None:
        if hr > 130 or hr < 45:
            escalate("RED", 9)
            alerts.append(f"Heart Rate {hr} bpm — Critical arrhythmia / tachycardia" if is_en else f"النبض {hr}/د — اضطراب نظم / تسارع حرج")
        elif hr > 110 or hr < 55:
            escalate("ORANGE", 6)
            alerts.append(f"Heart Rate {hr} bpm — Abnormal heart rate" if is_en else f"النبض {hr}/د — معدل نبض غير طبيعي")

    if si_val is not None:
        if si_val >= 1.3:
            escalate("RED", 9)
            alerts.append(f"Shock Index SI={si_val} (HR/SBP) — High occult shock risk" if is_en else f"مؤشر الصدمة SI={si_val} — اشتباه صدمة/نزيف حاد")
        elif si_val >= 1.0:
            escalate("ORANGE", 6)
            alerts.append(f"Shock Index SI={si_val} — Early shock warning" if is_en else f"مؤشر الصدمة SI={si_val} — مؤشر صدمة مبكر")

    if rr is not None:
        if rr > 28 or rr < 10:
            escalate("RED", 9)
            alerts.append(f"Respiratory Rate {rr}/min — Respiratory distress" if is_en else f"معدل التنفس {rr}/د — ضائقة تنفسية حادة")
        elif rr > 22:
            escalate("ORANGE", 6)
            alerts.append(f"Respiratory Rate {rr}/min — Tachypnea" if is_en else f"معدل التنفس {rr}/د — تسرع تنفس")

    if avpu in ("P", "U"):
        escalate("RED", 9)
        alerts.append(f"AVPU = {avpu} — Severe altered mental status" if is_en else f"مستوى الوعي AVPU={avpu} — تدهور شديد في الوعي")
    elif avpu == "V":
        escalate("ORANGE", 7)
        alerts.append("AVPU = V — Responds only to verbal stimuli" if is_en else "مستوى الوعي AVPU=V — يستجيب للصوت فقط")

    if gcs is not None:
        if gcs <= 8:
            escalate("RED", 10)
            alerts.append(f"GCS {gcs}/15 — Coma / Airway protection needed" if is_en else f"مقياس غلاسكو GCS={gcs}/15 — غيبوبة / تأمين مجرى الهواء")
        elif gcs <= 13:
            escalate("ORANGE", 7)
            alerts.append(f"GCS {gcs}/15 — Moderate neurological impairment" if is_en else f"مقياس غلاسكو GCS={gcs}/15 — اضطراب متوسط في الوعي")

    if temp is not None and (temp >= 39.5 or temp <= 35.0):
        escalate("ORANGE", 6)
        alerts.append(f"Temperature {temp}°C — Critical thermal abnormality" if is_en else f"الحرارة {temp}°C — اضطراب حراري حرج")

    if age is not None and age < 14 and tier != "GREEN":
        alerts.append("Pediatric patient (<14y) — PALS weight-based dosing required" if is_en else "مريض طفل (<14 سنة) — تطبيق بروتوكول PALS")

    if not alerts:
        alerts.append("All recorded vital signs within stable physiological limits." if is_en else "جميع العلامات الحيوية المسجلة ضمن الحدود المستقرة.")

    return {
        "tier": tier,
        "score": min_score if tier != "RED" else max(min_score, 9),
        "map": map_val,
        "shock_index": si_val,
        "alerts": alerts,
    }

# 5. دوال الذكاء الاصطناعي (Gemini Extraction, Audio Transcription & Clinical Draft)
def call_gemini_structured(contents: Any, schema_cls: Any, system_instruction: str):
    last_err = None
    for model_name in MODELS_PRIORITY:
        try:
            resp = client.models.generate_content(
                model=model_name,
                contents=contents,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    response_mime_type="application/json",
                    response_schema=schema_cls,
                    temperature=0.1,
                ),
            )
            parsed = schema_cls.model_validate_json(resp.text)
            return parsed, model_name
        except Exception as e:
            last_err = str(e)
            continue
    raise RuntimeError(last_err or "All Gemini models failed")

def api_extract_fields(text: str, lang: str = "en", mode: str = "deep") -> Dict[str, Any]:
    sys_prompt = (
        "You are a precision medical emergency data extractor for RELAY EMS Triage. "
        "Extract clinical fields accurately from Egyptian Arabic or English paramedic speech/notes. "
        "Rules: "
        "1) Normalize digits (Arabic/English) to standard integers/floats. "
        "2) Map gender strictly to 'Male' or 'Female' if stated, else null. "
        "3) Extract national_id as 14 digits without spaces if present. "
        "4) For medications, preserve both named drugs (e.g., Plavix, Insulin) AND unspecified drug classes "
        "(e.g., 'unknown blood thinners', 'unknown antiepileptic drugs'). "
        "5) Return null for any field not explicitly mentioned. Never guess or hallucinate."
    )
    parsed, model_used = call_gemini_structured(
        contents=f"MODE={mode}\nUI_LANG={lang}\nTRANSCRIPT:\n{text}",
        schema_cls=ExtractedCaseFields,
        system_instruction=sys_prompt,
    )
    return {"ok": True, "fields": parsed.model_dump(), "model": model_used}

def api_transcribe_audio(audio_b64: str, mime_type: str, lang: str = "en") -> Dict[str, Any]:
    raw_bytes = base64.b64decode(audio_b64)
    prompt = (
        "Transcribe this paramedic emergency voice handover verbatim. "
        f"Expected primary language: {'English' if lang == 'en' else 'Egyptian Arabic (with English medical terms)'}. "
        "Output ONLY the exact spoken transcript without commentary."
    )
    last_err = None
    for model_name in MODELS_PRIORITY:
        try:
            resp = client.models.generate_content(
                model=model_name,
                contents=[
                    types.Part.from_bytes(data=raw_bytes, mime_type=mime_type or "audio/webm"),
                    prompt,
                ],
                config=types.GenerateContentConfig(temperature=0.0),
            )
            transcript = (resp.text or "").strip()
            return {"ok": True, "transcript": transcript, "model": model_name}
        except Exception as e:
            last_err = str(e)
            continue
    return {"ok": False, "error": last_err or "Audio transcription failed"}

def api_generate_draft(payload: Dict[str, Any]) -> Dict[str, Any]:
    lang = payload.get("lang", "en")
    is_en = (lang == "en")
    vitals_eval = assess_vitals(payload, lang=lang)
    patient_id = generate_patient_id()
    safe_complaint = redact_phi_for_llm(payload.get("complaint", ""))

    out_lang_rule = (
        "CRITICAL LANGUAGE RULE: Write EVERY single field (clinical_summary, suggested_department, "
        "immediate_actions, staff_to_alert, equipment, labs_imaging, medications_prep, precautions) "
        "100% in clear, professional clinical ENGLISH. Do NOT include a single Arabic word or character."
        if is_en else
        "CRITICAL LANGUAGE RULE: Write EVERY single field (clinical_summary, suggested_department, "
        "immediate_actions, staff_to_alert, equipment, labs_imaging, medications_prep, precautions) "
        "100% in clear, professional clinical ARABIC (keeping standard drug/medical abbreviations in parentheses if helpful)."
    )

    sys_prompt = (
        "You are RELAY's clinical handover assistant generating a preliminary ER readiness draft. "
        f"{out_lang_rule} "
        "Keep lists concise (2-5 actionable items per category)."
    )

    clinical_context = {
        "age": payload.get("age"),
        "gender": payload.get("gender"),
        "vitals": {
            "SBP": payload.get("sbp"),
            "DBP": payload.get("dbp"),
            "MAP": vitals_eval["map"],
            "HR": payload.get("hr"),
            "Shock_Index": vitals_eval["shock_index"],
            "SpO2": payload.get("spo2"),
            "RR": payload.get("rr"),
            "Temp_C": payload.get("temp"),
            "AVPU": payload.get("avpu"),
            "GCS": payload.get("gcs"),
        },
        "deterministic_tier": vitals_eval["tier"],
        "deterministic_alerts": vitals_eval["alerts"],
        "medications": payload.get("medications"),
        "eta_minutes": payload.get("eta"),
        "redacted_complaint": safe_complaint,
    }

    try:
        draft_obj, model_used = call_gemini_structured(
            contents=json.dumps(clinical_context, ensure_ascii=False),
            schema_cls=ClinicalDraftResponse,
            system_instruction=sys_prompt,
        )
        draft_dict = draft_obj.model_dump()
    except Exception:
        model_used = "Rules-Only Fallback Engine"
        draft_dict = {
            "clinical_summary": (
                f"{payload.get('age') or '?'}y {payload.get('gender') or ''} patient, Tier {vitals_eval['tier']}, "
                f"BP {payload.get('sbp')}/{payload.get('dbp')}, HR {payload.get('hr')}, SpO2 {payload.get('spo2')}%, ETA {payload.get('eta')}m."
                if is_en else
                f"مريض عمر {payload.get('age') or '?'} سنة، الفئة {vitals_eval['tier']}، الضغط {payload.get('sbp')}/{payload.get('dbp')}، النبض {payload.get('hr')}، الأكسجين {payload.get('spo2')}%."
            ),
            "suggested_department": (
                "Emergency - Resuscitation & Critical Care Room" if (is_en and vitals_eval["tier"] == "RED")
                else ("Emergency - Acute Care Bay" if is_en else "الطوارئ - غرفة الإنعاش والحالات الحرجة")
            ),
            "immediate_actions": ["Airway & O2 check", "IV access", "Continuous cardiac monitor"] if is_en else ["تأمين مجرى الهواء", "تركيب خط وريدي", "توصيل مونيتور"],
            "staff_to_alert": ["Emergency Physician", "ER Resuscitation Nurse"] if is_en else ["طبيب طوارئ", "تمريض طوارئ"],
            "equipment": ["Multiparameter Monitor", "Defibrillator / Crash Cart"] if is_en else ["جهاز مونيتور", "عربة إنعاش"],
            "labs_imaging": ["CBC, Electrolytes, Blood Gas, ECG"] if is_en else ["تحاليل دم أساسية، غازات الدم، تخطيط قلب"],
            "medications_prep": ["Isotonic IV Crystalloids", "Oxygen"] if is_en else ["محلول وريدي", "أكسجين"],
            "precautions": vitals_eval["alerts"][:2],
        }

    return {
        "ok": True,
        "patient_id": patient_id,
        "model": model_used,
        "vitals_eval": vitals_eval,
        "draft": draft_dict,
    }

def send_telegram_alert(record: Dict[str, Any]) -> bool:
    if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
        return False
    is_en = (record.get("lang") == "en")
    tier_icon = {"RED": "🔴", "ORANGE": "🟠", "GREEN": "🟢"}.get(record["tier"], "⚪")
    if is_en:
        msg = (
            f"{tier_icon} *RELAY PRE-ARRIVAL ALERT ({record['tier']})*\n"
            f"🆔 *Patient ID:* `{record['patient_id']}`\n"
            f"👤 *Patient:* {record['patient_name']} | {record['age']}y | {record['gender']}\n"
            f"🪪 *National ID:* `{record['masked_nid']}`\n"
            f"⏱️ *ETA:* {record['eta']} min | 🏥 *Dept:* {record['department']}\n"
            f"🩺 *Vitals:* BP {record['sbp']}/{record['dbp']} (MAP {record['map']}) | HR {record['hr']} | SpO2 {record['spo2']}% | AVPU {record['avpu']} | GCS {record['gcs']}\n"
            f"💊 *Meds:* {record['medications'] or 'None'}\n"
            f"📋 *Summary:* {record['summary']}\n"
            f"✅ *Verified By:* {record['paramedic']} | 🤖 {record['model']}"
        )
    else:
        msg = (
            f"{tier_icon} *تنبيه RELAY قبل الوصول ({record['tier']})*\n"
            f"🆔 *رقم المريض:* `{record['patient_id']}`\n"
            f"👤 *المريض:* {record['patient_name']} | {record['age']} سنة | {record['gender']}\n"
            f"🪪 *الرقم القومي:* `{record['masked_nid']}`\n"
            f"⏱️ *الوصول المتوقع:* {record['eta']} دقيقة | 🏥 *القسم:* {record['department']}\n"
            f"🩺 *العلامات:* الضغط {record['sbp']}/{record['dbp']} (MAP {record['map']}) | النبض {record['hr']} | الأكسجين {record['spo2']}%\n"
            f"💊 *الأدوية:* {record['medications'] or 'لا يوجد'}\n"
            f"📋 *الملخص:* {record['summary']}\n"
            f"✅ *المراجع:* {record['paramedic']} | 🤖 {record['model']}"
        )
    try:
        r = requests.post(
            f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage",
            json={"chat_id": TELEGRAM_CHAT_ID, "text": msg, "parse_mode": "Markdown"},
            timeout=6,
        )
        return r.status_code == 200
    except Exception:
        return False

def api_confirm_case(record: Dict[str, Any]) -> Dict[str, Any]:
    record["masked_nid"] = mask_national_id(record.get("national_id", ""))
    record["confirmed_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    record.pop("national_id", None)  # لا نخزن الرقم القومي كاملاً
    with STORE_LOCK:
        CASES_STORE.insert(0, record)

    file_exists = os.path.isfile(LOG_FILE)
    with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["timestamp", "patient_id", "tier", "score", "patient_name", "age", "gender", "masked_nid", "sbp", "dbp", "map", "hr", "spo2", "eta", "department", "manually_edited", "paramedic", "model"])
        writer.writerow([
            record["confirmed_at"], record["patient_id"], record["tier"], record["score"],
            record["patient_name"], record["age"], record["gender"], record["masked_nid"],
            record["sbp"], record["dbp"], record["map"], record["hr"], record["spo2"],
            record["eta"], record["department"], record.get("manually_edited", False),
            record["paramedic"], record["model"]
        ])

    tg_sent = send_telegram_alert(record)
    return {"ok": True, "telegram_sent": tg_sent, "record": record}

# 6. واجهتا المستخدم الكاملتان (100% Bilingual Paramedic Terminal + /er Live Dashboard)
HTML_PAGE = r"""<!DOCTYPE html>
<html lang="en" dir="ltr">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<title>RELAY — Bridging the Gap Before Arrival</title>
<style>
  :root {
    --navy: #0f2942; --blue: #0284c7; --bg: #f1f5f9; --card: #ffffff;
    --border: #cbd5e1; --text: #0f172a; --muted: #475569;
    --red: #dc2626; --orange: #ea580c; --green: #16a34a;
  }
  * { box-sizing: border-box; font-family: 'Segoe UI', Tahoma, Arial, sans-serif; }
  body { margin: 0; background: var(--bg); color: var(--text); }
  .topbar {
    background: linear-gradient(135deg, #0b1f3a, #133b66); color: #fff;
    padding: 14px 22px; display: flex; justify-content: space-between; align-items: center;
    box-shadow: 0 3px 10px rgba(0,0,0,0.15);
  }
  .brand h1 { margin: 0; font-size: 22px; letter-spacing: 0.5px; }
  .brand small { color: #93c5fd; font-size: 12px; }
  .nav-actions { display: flex; gap: 10px; align-items: center; }
  .btn {
    border: none; border-radius: 8px; padding: 9px 15px; font-weight: 600;
    cursor: pointer; font-size: 13px; transition: 0.15s;
  }
  .btn:disabled { opacity: 0.6; cursor: not-allowed; }
  .btn-lang { background: #1e293b; color: #fff; border: 1px solid #475569; }
  .btn-er { background: #0284c7; color: #fff; }
  .btn-primary { background: #0284c7; color: #fff; width: 100%; font-size: 15px; padding: 12px; }
  .btn-dark { background: #0f172a; color: #fff; }
  .btn-danger { background: #dc2626; color: #fff; }
  .btn-outline { background: #fff; border: 1px solid var(--border); color: var(--text); }
  .container { max-width: 1120px; margin: 18px auto; padding: 0 16px 40px; }
  .card {
    background: var(--card); border: 1px solid var(--border); border-radius: 12px;
    padding: 16px 18px; margin-bottom: 14px; box-shadow: 0 1px 4px rgba(15,23,42,0.04);
  }
  .card h3 { margin: 0 0 12px; font-size: 15px; color: var(--navy); border-bottom: 1px solid #e2e8f0; padding-bottom: 8px; }
  .disclaimer {
    background: #fefce8; border: 1px solid #fde047; color: #854d0e;
    padding: 10px 14px; border-radius: 10px; font-size: 13px; margin-bottom: 12px;
  }
  .status-strip {
    background: #ecfdf5; border: 1px solid #6ee7b7; color: #065f46;
    padding: 8px 14px; border-radius: 8px; font-size: 12.5px; margin-bottom: 14px;
    display: flex; justify-content: space-between; flex-wrap: wrap; gap: 8px;
  }
  .voice-box {
    background: #e0f2fe; border: 1px solid #7dd3fc; border-radius: 12px;
    padding: 14px 16px; margin-bottom: 14px;
  }
  .interim-box {
    margin-top: 8px; background: #fff; border: 1px dashed #0284c7; border-radius: 8px;
    padding: 8px 12px; font-size: 13px; color: #0369a1; min-height: 36px;
  }
  .grid-4 { display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; }
  .grid-3 { display: grid; grid-template-columns: repeat(3, 1fr); gap: 12px; }
  .grid-2 { display: grid; grid-template-columns: repeat(2, 1fr); gap: 12px; }
  @media (max-width: 820px) { .grid-4, .grid-3, .grid-2 { grid-template-columns: 1fr; } }
  label { display: block; font-size: 12.5px; font-weight: 600; margin-bottom: 5px; color: #334155; }
  input, select, textarea {
    width: 100%; padding: 9px 11px; border: 1px solid var(--border); border-radius: 8px;
    font-size: 13.5px; background: #fff; color: var(--text);
  }
  input.edited, select.edited, textarea.edited { border-color: #d97706; background: #fffbeb; }
  .edit-tag { font-size: 11px; color: #b45309; font-weight: 600; display: none; margin-top: 2px; }
  .map-box {
    background: #f8fafc; border: 1px solid #cbd5e1; border-radius: 10px;
    padding: 10px; text-align: center; display: flex; flex-direction: column; justify-content: center;
  }
  .map-val { font-size: 18px; font-weight: 800; color: var(--navy); }
  .chip {
    background: #e0f2fe; color: #0369a1; border: 1px solid #bae6fd;
    padding: 5px 10px; border-radius: 6px; font-size: 12px; cursor: pointer; font-weight: 600;
  }
  .draft-banner {
    border-radius: 12px; padding: 16px; color: #fff; margin-top: 14px;
  }
  .tier-RED { background: linear-gradient(135deg, #b91c1c, #dc2626); }
  .tier-ORANGE { background: linear-gradient(135deg, #c2410c, #ea580c); }
  .tier-GREEN { background: linear-gradient(135deg, #15803d, #16a34a); }
  .review-box {
    background: #fffbeb; border: 2px solid #f59e0b; border-radius: 12px;
    padding: 16px; margin-top: 12px;
  }
  .prep-grid { display: grid; grid-template-columns: repeat(2, 1fr); gap: 10px; margin-top: 8px; }
  @media (max-width: 760px) { .prep-grid { grid-template-columns: 1fr; } }
  .prep-card { background: #fff; border: 1px solid #e2e8f0; border-radius: 8px; padding: 10px; font-size: 13px; }
  .er-card {
    background: #fff; border-radius: 12px; border: 1px solid var(--border);
    border-top: 6px solid var(--red); padding: 16px; margin-bottom: 14px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.06);
  }
  .er-card.ORANGE { border-top-color: var(--orange); }
  .er-card.GREEN { border-top-color: var(--green); }
  .badge { display: inline-block; padding: 4px 10px; border-radius: 999px; font-size: 12px; font-weight: 700; }
  .hidden { display: none !important; }
  table { width: 100%; border-collapse: collapse; font-size: 12.5px; }
  th, td { border-bottom: 1px solid #e2e8f0; padding: 8px; text-align: start; }
</style>
</head>
<body>

<div class="topbar">
  <div class="brand">
    <h1>🚑 RELAY</h1>
    <small id="t_tagline">Bridging the Gap Before Arrival | Pre-Hospital EMS Handover</small>
  </div>
  <div class="nav-actions">
    <button class="btn btn-lang" id="langToggleBtn" onclick="toggleLanguage()">العربية</button>
    <button class="btn btn-er" id="switchViewBtn" onclick="toggleView()">🏥 ER Hospital Live Dashboard</button>
  </div>
</div>

<!-- ================= VIEW 1: PARAMEDIC TERMINAL ================= -->
<div id="paramedicView" class="container">
  <div class="card" style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;">
    <div>
      <strong id="t_header_title">🚑 New Patient Handover</strong> &nbsp;|&nbsp;
      <span id="t_paramedic_lbl">Logged Paramedic:</span> <b>Paramedic 1 (EMS001)</b>
    </div>
    <button class="btn btn-outline" style="color:#dc2626; border-color:#fecaca;" id="t_clear_btn" onclick="resetCase()">🗑️ Clear Patient Case</button>
  </div>

  <div class="disclaimer" id="t_disclaimer">
    ⚠️ <b>Research Prototype — Not a standalone medical device.</b> Artificial Intelligence (AI) generates a preliminary draft only and may make errors; clinical verification and confirmation remain the responsibility of the attending paramedic.
  </div>

  <div class="status-strip">
    <span id="t_conn_status">✅ RELAY Connected | Active Models: gemini-2.5-flash → gemini-2.5-flash-lite</span>
    <span id="liveSyncNote"></span>
  </div>

  <!-- Live Voice + Two-Stage AI Extraction -->
  <div class="voice-box">
    <div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;">
      <div>
        <strong id="t_voice_title">🎙️ Live Voice Transcription + Real-Time AI Field Population</strong>
        <div id="t_voice_sub" style="font-size:12.5px; color:#0369a1; margin-top:3px;">
          Speak naturally in English or Egyptian Arabic: RELAY transcribes live, populates fields while you speak, and lets you run a deep verification pass anytime.
        </div>
      </div>
      <div style="display:flex; gap:8px; align-items:center;">
        <select id="speechLang" style="width:auto;">
          <option value="en-US">English (US)</option>
          <option value="ar-EG">العربية (مصر)</option>
        </select>
        <button class="btn btn-primary" style="width:auto;" id="recordBtn" onclick="toggleRecording()">🎤 Start Recording</button>
      </div>
    </div>
    <div class="interim-box" id="liveTranscriptBox">
      <span id="t_interim_placeholder">Live speech monitor ready — click "Start Recording" and speak your handover report...</span>
    </div>
  </div>

  <!-- 1. Patient Identification -->
  <div class="card">
    <h3 id="t_sec1">👤 1. Patient Identification</h3>
    <div class="grid-4">
      <div>
        <label id="t_lbl_name">Patient Name:</label>
        <input id="pName" placeholder="Enter full name" oninput="markEdited(this)" />
        <div class="edit-tag" data-edit-tag>✎ Manually edited</div>
      </div>
      <div>
        <label id="t_lbl_age">Age (years):</label>
        <input id="pAge" type="number" min="0" max="125" placeholder="Age (0 for infant)" oninput="markEdited(this)" />
        <div class="edit-tag" data-edit-tag>✎ Manually edited</div>
      </div>
      <div>
        <label id="t_lbl_gender">Gender:</label>
        <select id="pGender" onchange="markEdited(this)">
          <option value="" id="opt_g_unspec">Unspecified</option>
          <option value="Male" id="opt_g_male">Male</option>
          <option value="Female" id="opt_g_female">Female</option>
        </select>
        <div class="edit-tag" data-edit-tag>✎ Manually edited</div>
      </div>
      <div>
        <label id="t_lbl_nid">National ID (14 digits):</label>
        <input id="pNid" maxlength="14" placeholder="14 digits (masked in logs)" oninput="markEdited(this)" />
        <div class="edit-tag" data-edit-tag>✎ Manually edited</div>
      </div>
    </div>
  </div>

  <!-- 2. Main Vital Signs & MAP -->
  <div class="card">
    <h3 id="t_sec2">🩺 2. Main Vital Signs & Mean Arterial Pressure (MAP)</h3>
    <div class="grid-4" style="margin-bottom:12px;">
      <div>
        <label id="t_lbl_sbp">🩸 Systolic BP (SBP mmHg):</label>
        <input id="vSbp" type="number" placeholder="e.g. 120" oninput="markEdited(this); updateMap()" />
        <div class="edit-tag" data-edit-tag>✎ Manually edited</div>
      </div>
      <div>
        <label id="t_lbl_dbp">🩸 Diastolic BP (DBP mmHg):</label>
        <input id="vDbp" type="number" placeholder="e.g. 80" oninput="markEdited(this); updateMap()" />
        <div class="edit-tag" data-edit-tag>✎ Manually edited</div>
      </div>
      <div class="map-box">
        <div style="font-size:12px; font-weight:700;" id="t_map_title">Mean Arterial Pressure (MAP)</div>
        <div class="map-val" id="mapDisplay">--</div>
        <div style="font-size:11px; color:#64748b;" id="t_map_formula">DBP + (SBP - DBP) / 3</div>
      </div>
      <div>
        <label id="t_lbl_hr">💓 Heart Rate HR (bpm):</label>
        <input id="vHr" type="number" placeholder="e.g. 85" oninput="markEdited(this); updateMap()" />
        <div class="edit-tag" data-edit-tag>✎ Manually edited</div>
      </div>
    </div>
    <div class="grid-3">
      <div>
        <label id="t_lbl_spo2">🫁 Oxygen Saturation SpO₂ (%):</label>
        <input id="vSpo2" type="number" placeholder="e.g. 98" oninput="markEdited(this)" />
        <div class="edit-tag" data-edit-tag>✎ Manually edited</div>
      </div>
      <div>
        <label id="t_lbl_rr">🫁 Respiratory Rate RR (breaths/min):</label>
        <input id="vRr" type="number" placeholder="e.g. 16" oninput="markEdited(this)" />
        <div class="edit-tag" data-edit-tag>✎ Manually edited</div>
      </div>
      <div>
        <label id="t_lbl_temp">🌡️ Temperature (°C):</label>
        <input id="vTemp" type="number" step="0.1" placeholder="e.g. 37.0" oninput="markEdited(this)" />
        <div class="edit-tag" data-edit-tag>✎ Manually edited</div>
      </div>
    </div>
  </div>

  <!-- 3. Consciousness Assessment -->
  <div class="card">
    <h3 id="t_sec3">🧠 3. Consciousness Assessment (AVPU & GCS)</h3>
    <div class="grid-2">
      <div>
        <label id="t_lbl_avpu">AVPU Response Scale:</label>
        <select id="vAvpu" onchange="markEdited(this)">
          <option value="" id="opt_avpu_none">Not assessed</option>
          <option value="A" id="opt_avpu_a">A — Alert</option>
          <option value="V" id="opt_avpu_v">V — Responds to Voice</option>
          <option value="P" id="opt_avpu_p">P — Responds to Pain</option>
          <option value="U" id="opt_avpu_u">U — Unresponsive</option>
        </select>
        <div class="edit-tag" data-edit-tag>✎ Manually edited</div>
      </div>
      <div>
        <label id="t_lbl_gcs">Glasgow Coma Scale GCS (3–15):</label>
        <input id="vGcs" type="number" min="3" max="15" placeholder="Enter score (3–15)" oninput="markEdited(this)" />
        <div class="edit-tag" data-edit-tag>✎ Manually edited</div>
      </div>
    </div>
  </div>

  <!-- 4. Chief Complaint & Medications -->
  <div class="card">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:10px;">
      <h3 style="margin:0; border:none; padding:0;" id="t_sec4">📝 4. Chief Complaint & Current Medications</h3>
      <button class="btn btn-dark" id="deepExtractBtn" onclick="runExtraction('deep')">✨ Smart AI Extract (Deep Verify)</button>
    </div>
    <label id="t_lbl_complaint">Chief Complaint (Verbatim Raw Transcript — Preserved Completely):</label>
    <textarea id="vComplaint" rows="3" placeholder="Speak above or type clinical handover report here..."></textarea>
    <div style="margin-top:10px;">
      <label id="t_lbl_meds">💊 Current Medications (Named drugs & unspecified drug classes):</label>
      <input id="vMeds" placeholder="e.g. Plavix, Unknown blood thinners, Insulin" oninput="markEdited(this)" />
      <div class="edit-tag" data-edit-tag>✎ Manually edited</div>
    </div>
  </div>

  <!-- 5. ETA & Generate Draft -->
  <div class="card">
    <h3 id="t_sec5">⏱️ 5. Estimated Time of Arrival (ETA)</h3>
    <div style="display:flex; gap:10px; align-items:center; flex-wrap:wrap;">
      <input id="vEta" type="number" min="1" max="180" placeholder="Minutes (1–180)" style="max-width:220px;" oninput="markEdited(this)" />
      <button class="chip" onclick="setEta(5)">5m</button>
      <button class="chip" onclick="setEta(10)">10m</button>
      <button class="chip" onclick="setEta(15)">15m</button>
    </div>
    <div style="margin-top:14px;">
      <button class="btn btn-primary" id="genDraftBtn" onclick="generateDraft()">📝 Generate Draft for Review</button>
    </div>
  </div>

  <!-- 6. Human-in-the-Loop (HITL) Draft & Mandatory Confirmation Gate -->
  <div id="draftContainer" class="hidden">
    <div id="draftBanner" class="draft-banner tier-RED">
      <div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:8px;">
        <strong id="draftTierTitle" style="font-size:17px;">🔴 AI Preliminary Draft: RED Tier</strong>
        <span class="badge" style="background:rgba(0,0,0,0.28);" id="draftScoreBadge">Score: 10 / 10</span>
      </div>
      <div style="margin-top:6px; font-size:13px;" id="draftMetaLine"></div>
      <ul id="draftAlertsList" style="margin:10px 0 0; padding-inline-start:20px; font-size:13px;"></ul>
    </div>

    <div class="review-box">
      <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:10px;">
        <strong style="color:#92400e; font-size:15px;" id="t_hitl_title">✋ Mandatory Human-in-the-Loop Review Before Dispatch</strong>
        <span style="font-size:12px; color:#78350f;" id="draftModelBadge"></span>
      </div>

      <label id="t_hitl_sum_lbl">Suggested Clinical Summary (Editable):</label>
      <textarea id="rSummary" rows="2"></textarea>

      <div class="grid-3" style="margin-top:10px;">
        <div>
          <label id="t_hitl_tier_lbl">Confirmed Triage Tier:</label>
          <select id="rTier">
            <option value="RED" id="opt_t_red">🔴 RED — Immediate Resuscitation (9–10)</option>
            <option value="ORANGE" id="opt_t_orange">🟠 ORANGE — Urgent / High Risk (5–8)</option>
            <option value="GREEN" id="opt_t_green">🟢 GREEN — Stable / Standard Care (1–4)</option>
          </select>
        </div>
        <div>
          <label id="t_hitl_score_lbl">Severity Score (1–10):</label>
          <input id="rScore" type="number" min="1" max="10" />
        </div>
        <div>
          <label id="t_hitl_dept_lbl">Receiving Hospital Department:</label>
          <input id="rDept" />
        </div>
      </div>

      <div style="margin-top:12px;">
        <label id="t_hitl_prep_lbl">Suggested ER Readiness & Preparation Checklists:</label>
        <div class="prep-grid" id="prepGrid"></div>
      </div>

      <div style="margin-top:14px; background:#fff; padding:10px 12px; border-radius:8px; border:1px solid #fcd34d;">
        <label style="display:flex; gap:8px; align-items:center; cursor:pointer; margin:0;">
          <input type="checkbox" id="hitlConfirmCheck" style="width:18px; height:18px;" />
          <span id="t_hitl_ack">I have reviewed the raw transcript, vital signs, and clinical suggestions, and take clinical responsibility for confirming this handover.</span>
        </label>
      </div>

      <div style="display:flex; gap:10px; margin-top:12px;">
        <button class="btn btn-dark" style="flex:2; padding:12px; font-size:14.5px;" id="confirmSendBtn" onclick="confirmAndSend()">✅ Confirm, Save, and Dispatch to ER</button>
        <button class="btn btn-outline" style="flex:1;" id="cancelDraftBtn" onclick="document.getElementById('draftContainer').classList.add('hidden')">Cancel Draft</button>
      </div>
    </div>
  </div>

  <!-- 7. Audit Log Table -->
  <div class="card" style="margin-top:16px;">
    <h3 id="t_log_title">📋 Verified & Dispatched Handover Audit Log (PHI Masked)</h3>
    <div style="overflow-x:auto;">
      <table>
        <thead>
          <tr>
            <th id="th_id">Patient ID</th>
            <th id="th_tier">Tier</th>
            <th id="th_pat">Patient</th>
            <th id="th_nid">Masked ID</th>
            <th id="th_vit">Vitals / MAP</th>
            <th id="th_eta">ETA</th>
            <th id="th_ed">Human Edited</th>
          </tr>
        </thead>
        <tbody id="auditTableBody"></tbody>
      </table>
    </div>
  </div>
</div>

<!-- ================= VIEW 2: HOSPITAL ER LIVE DASHBOARD ================= -->
<div id="erView" class="container hidden">
  <div class="card" style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;">
    <div>
      <strong style="font-size:17px;" id="t_er_title">🏥 Hospital Emergency Room (ER) Live Dashboard</strong>
      <div style="font-size:12.5px; color:#475569;" id="t_er_sub">Real-time pre-arrival telemetry & resuscitation room readiness</div>
    </div>
    <div style="display:flex; gap:8px; align-items:center;">
      <span class="badge" style="background:#dcfce7; color:#166534;" id="t_er_live">● Live Sync Active</span>
      <button class="btn btn-outline" id="chimeBtn" onclick="toggleChime()">🔔 Audio Alert: ON</button>
    </div>
  </div>

  <div class="grid-4" style="margin-bottom:14px;">
    <div class="card" style="text-align:center; border-top:4px solid #0284c7;">
      <div style="font-size:12px; color:#475569;" id="t_cnt_tot">Total Incoming Cases</div>
      <div style="font-size:26px; font-weight:800;" id="cntTotal">0</div>
    </div>
    <div class="card" style="text-align:center; border-top:4px solid #dc2626;">
      <div style="font-size:12px; color:#dc2626;" id="t_cnt_red">🔴 RED (Immediate Resuscitation)</div>
      <div style="font-size:26px; font-weight:800; color:#dc2626;" id="cntRed">0</div>
    </div>
    <div class="card" style="text-align:center; border-top:4px solid #ea580c;">
      <div style="font-size:12px; color:#ea580c;" id="t_cnt_org">🟠 ORANGE (Urgent)</div>
      <div style="font-size:26px; font-weight:800; color:#ea580c;" id="cntOrange">0</div>
    </div>
    <div class="card" style="text-align:center; border-top:4px solid #16a34a;">
      <div style="font-size:12px; color:#16a34a;" id="t_cnt_grn">🟢 GREEN (Stable)</div>
      <div style="font-size:26px; font-weight:800; color:#16a34a;" id="cntGreen">0</div>
    </div>
  </div>

  <div id="erCasesList"></div>
</div>

<script>
  let uiLang = 'en';
  let currentView = window.location.pathname === '/er' ? 'er' : 'paramedic';
  let isRecording = false;
  let recognition = null;
  let mediaRecorder = null;
  let audioChunks = [];
  let liveExtractTimer = null;
  let lastExtractedSnapshot = '';
  let currentDraft = null;
  let anyManualEdit = false;
  let chimeEnabled = true;
  let knownCaseIds = new Set();

  const I18N = {
    en: {
      dir: 'ltr', langToggle: 'العربية',
      switchToEr: '🏥 ER Hospital Live Dashboard', switchToParam: '🚑 Paramedic Intake Terminal',
      tagline: 'Bridging the Gap Before Arrival | Pre-Hospital EMS Handover',
      headerTitle: '🚑 New Patient Handover', paramedicLbl: 'Logged Paramedic:',
      clearBtn: '🗑️ Clear Patient Case',
      disclaimer: '⚠️ <b>Research Prototype — Not a standalone medical device.</b> Artificial Intelligence (AI) generates a preliminary draft only and may make errors; clinical verification and confirmation remain the responsibility of the attending paramedic.',
      connStatus: '✅ RELAY Connected | Active Models: gemini-2.5-flash → gemini-2.5-flash-lite',
      voiceTitle: '🎙️ Live Voice Transcription + Real-Time AI Field Population',
      voiceSub: 'Speak naturally in English or Egyptian Arabic: RELAY transcribes live, populates fields while you speak, and lets you run a deep verification pass anytime.',
      recStart: '🎤 Start Recording', recStop: '⏹️ Stop Recording',
      interimReady: 'Live speech monitor ready — click "Start Recording" and speak your handover report...',
      sec1: '👤 1. Patient Identification', lblName: 'Patient Name:', phName: 'Enter full name',
      lblAge: 'Age (years):', phAge: 'Age (0 for infant)', lblGender: 'Gender:',
      gUnspec: 'Unspecified', gMale: 'Male', gFemale: 'Female',
      lblNid: 'National ID (14 digits):', phNid: '14 digits (masked in logs)',
      sec2: '🩺 2. Main Vital Signs & Mean Arterial Pressure (MAP)',
      lblSbp: '🩸 Systolic BP (SBP mmHg):', lblDbp: '🩸 Diastolic BP (DBP mmHg):',
      mapTitle: 'Mean Arterial Pressure (MAP)', lblHr: '💓 Heart Rate HR (bpm):',
      lblSpo2: '🫁 Oxygen Saturation SpO₂ (%):', lblRr: '🫁 Respiratory Rate RR (breaths/min):',
      lblTemp: '🌡️ Temperature (°C):',
      sec3: '🧠 3. Consciousness Assessment (AVPU & GCS)',
      lblAvpu: 'AVPU Response Scale:', avpuNone: 'Not assessed', avpuA: 'A — Alert',
      avpuV: 'V — Responds to Voice', avpuP: 'P — Responds to Pain', avpuU: 'U — Unresponsive',
      lblGcs: 'Glasgow Coma Scale GCS (3–15):', phGcs: 'Enter score (3–15)',
      sec4: '📝 4. Chief Complaint & Current Medications',
      deepBtn: '✨ Smart AI Extract (Deep Verify)',
      lblComplaint: 'Chief Complaint (Verbatim Raw Transcript — Preserved Completely):',
      phComplaint: 'Speak above or type clinical handover report here...',
      lblMeds: '💊 Current Medications (Named drugs & unspecified drug classes):',
      phMeds: 'e.g. Plavix, Unknown blood thinners, Insulin',
      sec5: '⏱️ 5. Estimated Time of Arrival (ETA)', phEta: 'Minutes (1–180)',
      genBtn: '📝 Generate Draft for Review',
      hitlTitle: '✋ Mandatory Human-in-the-Loop Review Before Dispatch',
      hitlSumLbl: 'Suggested Clinical Summary (Editable):',
      hitlTierLbl: 'Confirmed Triage Tier:', hitlScoreLbl: 'Severity Score (1–10):',
      hitlDeptLbl: 'Receiving Hospital Department:',
      tRed: '🔴 RED — Immediate Resuscitation (9–10)',
      tOrange: '🟠 ORANGE — Urgent / High Risk (5–8)',
      tGreen: '🟢 GREEN — Stable / Standard Care (1–4)',
      hitlPrepLbl: 'Suggested ER Readiness & Preparation Checklists:',
      hitlAck: 'I have reviewed the raw transcript, vital signs, and clinical suggestions, and take clinical responsibility for confirming this handover.',
      confirmBtn: '✅ Confirm, Save, and Dispatch to ER', cancelBtn: 'Cancel Draft',
      logTitle: '📋 Verified & Dispatched Handover Audit Log (PHI Masked)',
      thId: 'Patient ID', thTier: 'Tier', thPat: 'Patient', thNid: 'Masked ID',
      thVit: 'Vitals / MAP', thEta: 'ETA', thEd: 'Human Edited',
      erTitle: '🏥 Hospital Emergency Room (ER) Live Dashboard',
      erSub: 'Real-time pre-arrival telemetry & resuscitation room readiness',
      erLive: '● Live Sync Active',
      cntTot: 'Total Incoming Cases', cntRed: '🔴 RED (Immediate Resuscitation)',
      cntOrg: '🟠 ORANGE (Urgent)', cntGrn: '🟢 GREEN (Stable)',
      noCases: '🚑 No incoming cases yet. Confirmed ambulance handovers will appear here instantaneously.',
      editTag: '✎ Manually edited',
      prepCats: {
        immediate_actions: '🚑 Immediate Actions',
        staff_to_alert: '👨‍⚕️ Staff to Alert',
        equipment: '🩺 Equipment & Devices',
        labs_imaging: '🔬 Labs & Imaging',
        medications_prep: '💊 Medications to Prepare',
        precautions: '⚠️ Clinical Precautions'
      }
    },
    ar: {
      dir: 'rtl', langToggle: 'English',
      switchToEr: '🏥 شاشة استقبال الطوارئ (ER Dashboard)', switchToParam: '🚑 واجهة تسجيل المسعف',
      tagline: 'سد الفجوة قبل الوصول | تسليم حالات الطوارئ ما قبل المستشفى',
      headerTitle: '🚑 تسجيل حالة جديدة', paramedicLbl: 'المسعف المسجل:',
      clearBtn: '🗑️ مسح بيانات الحالة',
      disclaimer: '⚠️ <b>نسخة تجريبية — ليست أداة قرار سريري مستقلة.</b> الذكاء الاصطناعي يقترح مسودة أولية فقط وقد يخطئ، والمراجعة والتأكيد مسؤولية المسعف المختص.',
      connStatus: '✅ RELAY متصل | النماذج النشطة: gemini-2.5-flash → gemini-2.5-flash-lite',
      voiceTitle: '🎙️ التفريغ الصوتي الحي + التعبئة الذكية الفورية للخانات',
      voiceSub: 'تحدث بصورة طبيعية بالعربية أو الإنجليزية: يقوم النظام بالتفريغ الحي وتعبئة الخانات أثناء حديثك، مع زر تدقيق عميق إضافي.',
      recStart: '🎤 ابدأ التسجيل', recStop: '⏹️ إيقاف التسجيل',
      interimReady: 'مراقب الصوت الحي جاهز — اضغط "ابدأ التسجيل" وتحدث بتقرير الحالة...',
      sec1: '👤 1. بيانات تعريف المريض', lblName: 'اسم المريض:', phName: 'أدخل الاسم الثلاثي',
      lblAge: 'العمر (سنة):', phAge: 'العمر (0 للرضع)', lblGender: 'النوع (Gender):',
      gUnspec: 'غير محدد', gMale: 'ذكر (Male)', gFemale: 'أنثى (Female)',
      lblNid: 'الرقم القومي (14 رقماً):', phNid: '14 رقماً (يُحجب في السجلات)',
      sec2: '🩺 2. العلامات الحيوية ومتوسط الضغط الشرياني (MAP)',
      lblSbp: '🩸 الضغط الانقباضي (SBP mmHg):', lblDbp: '🩸 الضغط الانبساطي (DBP mmHg):',
      mapTitle: 'متوسط الضغط الشرياني (MAP)', lblHr: '💓 النبض HR (نبضة/د):',
      lblSpo2: '🫁 الأكسجين SpO₂ (%):', lblRr: '🫁 معدل التنفس RR (نفس/د):',
      lblTemp: '🌡️ درجة الحرارة (°C):',
      sec3: '🧠 3. تقييم مستوى الوعي (AVPU & GCS)',
      lblAvpu: 'مقياس الاستجابة AVPU:', avpuNone: 'لم يُقيّم', avpuA: 'A — واعٍ ومنتبه',
      avpuV: 'V — يستجيب للصوت', avpuP: 'P — يستجيب للألم فقط', avpuU: 'U — غير مستجيب',
      lblGcs: 'مقياس غلاسكو للغيبوبة GCS (3–15):', phGcs: 'أدخل الدرجة (3–15)',
      sec4: '📝 4. الشكوى الرئيسية والأدوية الحالية',
      deepBtn: '✨ استخراج ذكي من النص (تدقيق عميق)',
      lblComplaint: 'الشكوى الرئيسية (السجل الصوتي الحرفي الكامل — يُحفظ دون حذف):',
      phComplaint: 'تحدث بالأعلى أو اكتب تقرير الحالة هنا...',
      lblMeds: '💊 الأدوية الحالية (الأسماء التجارية أو الفئات الدوائية غير المعروفة):',
      phMeds: 'مثال: Plavix، مسيلات دم غير معروفة، أنسولين',
      sec5: '⏱️ 5. وقت الوصول المتوقع للمستشفى (ETA)', phEta: 'الوقت بالدقائق (1–180)',
      genBtn: '📝 إنشاء مسودة للمراجعة',
      hitlTitle: '✋ بوابة المراجعة البشرية الإلزامية قبل الإرسال',
      hitlSumLbl: 'الملخص السريري المقترح (قابل للتعديل):',
      hitlTierLbl: 'الفئة المعتمدة:', hitlScoreLbl: 'درجة الخطورة (1–10):',
      hitlDeptLbl: 'القسم المستهدف بالمستشفى:',
      tRed: '🔴 أحمر — إنعاش فوري (9–10)',
      tOrange: '🟠 برتقالي — عاجلة عالية الخطورة (5–8)',
      tGreen: '🟢 أخضر — مستقرة (1–4)',
      hitlPrepLbl: 'التجهيزات السريرية المقترحة قبل الوصول:',
      hitlAck: 'راجعت النص الحرفي والعلامات الحيوية والاقتراحات السريرية، وأتحمل مسؤولية تأكيد بيانات التسليم.',
      confirmBtn: '✅ تأكيد وحفظ وإرسال البلاغ للطوارئ', cancelBtn: 'إلغاء المسودة',
      logTitle: '📋 سجل الحالات المؤكدة والمرسلة (مع حجب الهوية)',
      thId: 'رقم المريض', thTier: 'الفئة', thPat: 'المريض', thNid: 'الرقم القومي المحجوب',
      thVit: 'العلامات / MAP', thEta: 'الوصول', thEd: 'تعديل بشري',
      erTitle: '🏥 لوحة استقبال طوارئ المستشفى الحية (ER Dashboard)',
      erSub: 'متابعة بلاغات الإسعاف لحظياً وتجهيز غرف الإنعاش قبل الوصول',
      erLive: '● متصل لحظياً',
      cntTot: 'إجمالي الحالات الواردة', cntRed: '🔴 حالات حمراء (إنعاش فوري)',
      cntOrg: '🟠 حالات برتقالية (عاجلة)', cntGrn: '🟢 حالات خضراء (مستقرة)',
      noCases: '🚑 لا توجد حالات واردة حالياً. ستظهر أي حالة يؤكدها المسعف هنا فوراً.',
      editTag: '✎ تم التعديل يدوياً',
      prepCats: {
        immediate_actions: '🚑 إجراءات فورية',
        staff_to_alert: '👨‍⚕️ تخصصات يجب إخطارها',
        equipment: '🩺 معدات وأجهزة',
        labs_imaging: '🔬 فحوصات وأشعة',
        medications_prep: '💊 أدوية للتجهيز',
        precautions: '⚠️ احتياطات سريرية'
      }
    }
  };

  function applyLanguage() {
    const t = I18N[uiLang];
    document.documentElement.lang = uiLang;
    document.documentElement.dir = t.dir;
    document.getElementById('langToggleBtn').textContent = t.langToggle;
    document.getElementById('switchViewBtn').textContent = currentView === 'paramedic' ? t.switchToEr : t.switchToParam;
    document.getElementById('t_tagline').textContent = t.tagline;
    document.getElementById('t_header_title').textContent = t.headerTitle;
    document.getElementById('t_paramedic_lbl').textContent = t.paramedicLbl;
    document.getElementById('t_clear_btn').textContent = t.clearBtn;
    document.getElementById('t_disclaimer').innerHTML = t.disclaimer;
    document.getElementById('t_conn_status').textContent = t.connStatus;
    document.getElementById('t_voice_title').textContent = t.voiceTitle;
    document.getElementById('t_voice_sub').textContent = t.voiceSub;
    document.getElementById('recordBtn').textContent = isRecording ? t.recStop : t.recStart;
    if (!isRecording) document.getElementById('liveTranscriptBox').textContent = t.interimReady;
    document.getElementById('speechLang').value = uiLang === 'en' ? 'en-US' : 'ar-EG';

    document.getElementById('t_sec1').textContent = t.sec1;
    document.getElementById('t_lbl_name').textContent = t.lblName;
    document.getElementById('pName').placeholder = t.phName;
    document.getElementById('t_lbl_age').textContent = t.lblAge;
    document.getElementById('pAge').placeholder = t.phAge;
    document.getElementById('t_lbl_gender').textContent = t.lblGender;
    document.getElementById('opt_g_unspec').textContent = t.gUnspec;
    document.getElementById('opt_g_male').textContent = t.gMale;
    document.getElementById('opt_g_female').textContent = t.gFemale;
    document.getElementById('t_lbl_nid').textContent = t.lblNid;
    document.getElementById('pNid').placeholder = t.phNid;

    document.getElementById('t_sec2').textContent = t.sec2;
    document.getElementById('t_lbl_sbp').textContent = t.lblSbp;
    document.getElementById('t_lbl_dbp').textContent = t.lblDbp;
    document.getElementById('t_map_title').textContent = t.mapTitle;
    document.getElementById('t_lbl_hr').textContent = t.lblHr;
    document.getElementById('t_lbl_spo2').textContent = t.lblSpo2;
    document.getElementById('t_lbl_rr').textContent = t.lblRr;
    document.getElementById('t_lbl_temp').textContent = t.lblTemp;

    document.getElementById('t_sec3').textContent = t.sec3;
    document.getElementById('t_lbl_avpu').textContent = t.lblAvpu;
    document.getElementById('opt_avpu_none').textContent = t.avpuNone;
    document.getElementById('opt_avpu_a').textContent = t.avpuA;
    document.getElementById('opt_avpu_v').textContent = t.avpuV;
    document.getElementById('opt_avpu_p').textContent = t.avpuP;
    document.getElementById('opt_avpu_u').textContent = t.avpuU;
    document.getElementById('t_lbl_gcs').textContent = t.lblGcs;
    document.getElementById('vGcs').placeholder = t.phGcs;

    document.getElementById('t_sec4').textContent = t.sec4;
    document.getElementById('deepExtractBtn').textContent = t.deepBtn;
    document.getElementById('t_lbl_complaint').textContent = t.lblComplaint;
    document.getElementById('vComplaint').placeholder = t.phComplaint;
    document.getElementById('t_lbl_meds').textContent = t.lblMeds;
    document.getElementById('vMeds').placeholder = t.phMeds;

    document.getElementById('t_sec5').textContent = t.sec5;
    document.getElementById('vEta').placeholder = t.phEta;
    document.getElementById('genDraftBtn').textContent = t.genBtn;

    document.getElementById('t_hitl_title').textContent = t.hitlTitle;
    document.getElementById('t_hitl_sum_lbl').textContent = t.hitlSumLbl;
    document.getElementById('t_hitl_tier_lbl').textContent = t.hitlTierLbl;
    document.getElementById('t_hitl_score_lbl').textContent = t.hitlScoreLbl;
    document.getElementById('t_hitl_dept_lbl').textContent = t.hitlDeptLbl;
    document.getElementById('opt_t_red').textContent = t.tRed;
    document.getElementById('opt_t_orange').textContent = t.tOrange;
    document.getElementById('opt_t_green').textContent = t.tGreen;
    document.getElementById('t_hitl_prep_lbl').textContent = t.hitlPrepLbl;
    document.getElementById('t_hitl_ack').textContent = t.hitlAck;
    document.getElementById('confirmSendBtn').textContent = t.confirmBtn;
    document.getElementById('cancelDraftBtn').textContent = t.cancelBtn;

    document.getElementById('t_log_title').textContent = t.logTitle;
    document.getElementById('th_id').textContent = t.thId;
    document.getElementById('th_tier').textContent = t.thTier;
    document.getElementById('th_pat').textContent = t.thPat;
    document.getElementById('th_nid').textContent = t.thNid;
    document.getElementById('th_vit').textContent = t.thVit;
    document.getElementById('th_eta').textContent = t.thEta;
    document.getElementById('th_ed').textContent = t.thEd;

    document.getElementById('t_er_title').textContent = t.erTitle;
    document.getElementById('t_er_sub').textContent = t.erSub;
    document.getElementById('t_er_live').textContent = t.erLive;
    document.getElementById('t_cnt_tot').textContent = t.cntTot;
    document.getElementById('t_cnt_red').textContent = t.cntRed;
    document.getElementById('t_cnt_org').textContent = t.cntOrg;
    document.getElementById('t_cnt_grn').textContent = t.cntGrn;
    document.getElementById('chimeBtn').textContent = uiLang === 'en'
      ? `🔔 Audio Alert: ${chimeEnabled ? 'ON' : 'OFF'}`
      : `🔔 التنبيه الصوتي: ${chimeEnabled ? 'مفعّل' : 'مغلق'}`;

    document.querySelectorAll('[data-edit-tag]').forEach(el => el.textContent = t.editTag);
    refreshCases();
  }

  function toggleLanguage() {
    uiLang = uiLang === 'en' ? 'ar' : 'en';
    applyLanguage();
  }

  function toggleView() {
    currentView = currentView === 'paramedic' ? 'er' : 'paramedic';
    document.getElementById('paramedicView').classList.toggle('hidden', currentView !== 'paramedic');
    document.getElementById('erView').classList.toggle('hidden', currentView !== 'er');
    applyLanguage();
  }

  // --- LIVE SPEECH RECOGNITION + LIVE AI POPULATION + AUDIO FALLBACK ---
  async function toggleRecording() {
    if (isRecording) {
      stopRecording();
    } else {
      await startRecording();
    }
  }

  async function startRecording() {
    const t = I18N[uiLang];
    const box = document.getElementById('liveTranscriptBox');
    isRecording = true;
    document.getElementById('recordBtn').textContent = t.recStop;
    document.getElementById('recordBtn').classList.remove('btn-primary');
    document.getElementById('recordBtn').classList.add('btn-danger');
    box.innerHTML = uiLang === 'en' ? '🎙️ <b>Listening live...</b> Speak clearly now.' : '🎙️ <b>جاري الاستماع الحي...</b> تحدث بوضوح الآن.';

    // 1. بدء تسجيل صوتي احتياطي عبر MediaRecorder
    audioChunks = [];
    try {
      const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
      mediaRecorder = new MediaRecorder(stream);
      mediaRecorder.ondataavailable = e => { if (e.data.size > 0) audioChunks.push(e.data); };
      mediaRecorder.onstop = async () => {
        stream.getTracks().forEach(tr => tr.stop());
        // إذا لم يلتقط التعرف الصوتي للمتصفح نصاً، نرسله لـ Gemini Audio تلقائياً
        if (document.getElementById('vComplaint').value.trim().length < 5 && audioChunks.length > 0) {
          box.textContent = uiLang === 'en' ? '⏳ Transcribing recorded audio via Gemini...' : '⏳ جاري تفريغ التسجيل الصوتي عبر Gemini...';
          const blob = new Blob(audioChunks, { type: mediaRecorder.mimeType || 'audio/webm' });
          const reader = new FileReader();
          reader.onloadend = async () => {
            const b64 = reader.result.split(',')[1];
            const res = await fetch('/api/transcribe_audio', {
              method: 'POST', headers: {'Content-Type': 'application/json'},
              body: JSON.stringify({ audio_b64: b64, mime_type: blob.type, lang: uiLang })
            }).then(r => r.json());
            if (res.ok && res.transcript) {
              document.getElementById('vComplaint').value = res.transcript;
              box.textContent = '✅ ' + res.transcript;
              runExtraction('deep');
            } else {
              box.textContent = '⚠️ ' + (res.error || 'No speech captured');
            }
          };
          reader.readAsDataURL(blob);
        }
      };
      mediaRecorder.start();
    } catch (micErr) {
      box.textContent = (uiLang === 'en' ? '⚠️ Microphone permission notice: ' : '⚠️ تنبيه إذن الميكروفون: ') + micErr.message;
    }

    // 2. بدء التفريغ الحي الفوري Web Speech API
    const SR = window.SpeechRecognition || window.webkitSpeechRecognition;
    if (SR) {
      recognition = new SR();
      recognition.lang = document.getElementById('speechLang').value;
      recognition.continuous = true;
      recognition.interimResults = true;

      recognition.onresult = (event) => {
        let interim = '';
        let finalChunk = '';
        for (let i = event.resultIndex; i < event.results.length; i++) {
          const piece = event.results[i][0].transcript;
          if (event.results[i].isFinal) {
            finalChunk += piece + ' ';
          } else {
            interim += piece;
          }
        }
        if (finalChunk.trim()) {
          const ta = document.getElementById('vComplaint');
          ta.value = (ta.value + ' ' + finalChunk).trim();
          scheduleLiveExtraction();
        }
        box.innerHTML = (uiLang === 'en' ? '🎙️ <b>Live:</b> ' : '🎙️ <b>مباشر:</b> ') +
          (interim ? `<i>${interim}</i>` : document.getElementById('vComplaint').value);
      };

      recognition.onerror = (e) => {
        if (e.error !== 'no-speech' && e.error !== 'aborted') {
          box.textContent = (uiLang === 'en' ? '⚠️ Browser live speech fallback active: ' : '⚠️ تفعيل المسار الصوتي البديل: ') + e.error;
        }
      };

      recognition.onend = () => {
        if (isRecording) {
          try { recognition.start(); } catch (_) {}
        }
      };

      try { recognition.start(); } catch (_) {}
    }
  }

  function stopRecording() {
    isRecording = false;
    const t = I18N[uiLang];
    const btn = document.getElementById('recordBtn');
    btn.textContent = t.recStart;
    btn.classList.remove('btn-danger');
    btn.classList.add('btn-primary');
    if (recognition) { try { recognition.stop(); } catch (_) {} }
    if (mediaRecorder && mediaRecorder.state !== 'inactive') { try { mediaRecorder.stop(); } catch (_) {} }
    if (document.getElementById('vComplaint').value.trim().length >= 5) {
      runExtraction('deep');
    }
  }

  function scheduleLiveExtraction() {
    clearTimeout(liveExtractTimer);
    liveExtractTimer = setTimeout(() => {
      const txt = document.getElementById('vComplaint').value.trim();
      if (txt.length >= 12 && txt !== lastExtractedSnapshot) {
        lastExtractedSnapshot = txt;
        runExtraction('fast');
      }
    }, 2200);
  }

  function applyFieldIfUnedited(id, val) {
    if (val === null || val === undefined || val === '') return;
    const el = document.getElementById(id);
    if (!el || el.dataset.manual === '1') return;
    el.value = val;
  }

  async function runExtraction(mode) {
    const txt = document.getElementById('vComplaint').value.trim();
    const note = document.getElementById('liveSyncNote');
    if (txt.length < 4) return;
    const btn = document.getElementById('deepExtractBtn');
    if (mode === 'deep') btn.disabled = true;
    note.textContent = uiLang === 'en'
      ? (mode === 'fast' ? '⚡ Live AI populating fields...' : '🧠 Deep AI verification in progress...')
      : (mode === 'fast' ? '⚡ تعبئة حية للخانات...' : '🧠 جاري التدقيق العميق بالذكاء الاصطناعي...');

    try {
      const res = await fetch('/api/extract', {
        method: 'POST', headers: {'Content-Type': 'application/json'},
        body: JSON.stringify({ text: txt, lang: uiLang, mode })
      }).then(r => r.json());

      if (res.ok && res.fields) {
        const f = res.fields;
        applyFieldIfUnedited('pName', f.patient_name);
        applyFieldIfUnedited('pAge', f.age);
        if (f.gender) {
          const g = String(f.gender).toLowerCase();
          applyFieldIfUnedited('pGender', g.startsWith('f') ? 'Female' : (g.startsWith('m') ? 'Male' : ''));
        }
        applyFieldIfUnedited('pNid', f.national_id);
        applyFieldIfUnedited('vSbp', f.sbp);
        applyFieldIfUnedited('vDbp', f.dbp);
        applyFieldIfUnedited('vHr', f.hr);
        applyFieldIfUnedited('vSpo2', f.spo2);
        applyFieldIfUnedited('vRr', f.rr);
        applyFieldIfUnedited('vTemp', f.temp);
        applyFieldIfUnedited('vAvpu', f.avpu);
        applyFieldIfUnedited('vGcs', f.gcs);
        applyFieldIfUnedited('vMeds', f.medications);
        applyFieldIfUnedited('vEta', f.eta);
        updateMap();
        note.textContent = uiLang === 'en' ? `✅ Fields synced (${res.model})` : `✅ تم تحديث الخانات (${res.model})`;
      }
    } catch (e) {
      note.textContent = '⚠️ ' + e.message;
    } finally {
      if (mode === 'deep') btn.disabled = false;
    }
  }

  function markEdited(el) {
    el.dataset.manual = '1';
    el.classList.add('edited');
    anyManualEdit = true;
    const tag = el.parentElement.querySelector('[data-edit-tag]');
    if (tag) tag.style.display = 'block';
  }

  function updateMap() {
    const sbp = parseFloat(document.getElementById('vSbp').value);
    const dbp = parseFloat(document.getElementById('vDbp').value);
    const hr = parseFloat(document.getElementById('vHr').value);
    const disp = document.getElementById('mapDisplay');
    if (!isNaN(sbp) && !isNaN(dbp) && sbp > 0 && dbp > 0) {
      const map = Math.round(dbp + (sbp - dbp) / 3);
      const si = (!isNaN(hr) && hr > 0) ? ` | SI ${(hr / sbp).toFixed(2)}` : '';
      disp.textContent = `${map} mmHg${si}`;
      disp.style.color = map < 65 ? '#dc2626' : '#0f2942';
    } else {
      disp.textContent = '--';
    }
  }

  function setEta(mins) {
    const el = document.getElementById('vEta');
    el.value = mins;
    markEdited(el);
  }

  function resetCase() {
    ['pName','pAge','pGender','pNid','vSbp','vDbp','vHr','vSpo2','vRr','vTemp','vAvpu','vGcs','vComplaint','vMeds','vEta'].forEach(id => {
      const el = document.getElementById(id);
      el.value = '';
      el.dataset.manual = '0';
      el.classList.remove('edited');
    });
    document.querySelectorAll('[data-edit-tag]').forEach(el => el.style.display = 'none');
    document.getElementById('draftContainer').classList.add('hidden');
    anyManualEdit = false;
    lastExtractedSnapshot = '';
    updateMap();
  }

  function numOrNull(id) {
    const v = document.getElementById(id).value.trim();
    return v === '' ? null : Number(v);
  }

  async function generateDraft() {
    const btn = document.getElementById('genDraftBtn');
    btn.disabled = true;
    btn.textContent = uiLang === 'en' ? '⏳ Running Clinical Rules & AI Draft...' : '⏳ جاري تطبيق القواعد السريرية وإنشاء المسودة...';

    const payload = {
      lang: uiLang,
      patient_name: document.getElementById('pName').value.trim() || (uiLang === 'en' ? 'Unidentified Patient' : 'مريض غير محدد الاسم'),
      age: numOrNull('pAge'),
      gender: document.getElementById('pGender').value || (uiLang === 'en' ? 'Unspecified' : 'غير محدد'),
      national_id: document.getElementById('pNid').value.trim(),
      sbp: numOrNull('vSbp'),
      dbp: numOrNull('vDbp'),
      hr: numOrNull('vHr'),
      spo2: numOrNull('vSpo2'),
      rr: numOrNull('vRr'),
      temp: numOrNull('vTemp'),
      avpu: document.getElementById('vAvpu').value,
      gcs: numOrNull('vGcs'),
      complaint: document.getElementById('vComplaint').value.trim(),
      medications: document.getElementById('vMeds').value.trim(),
      eta: numOrNull('vEta') || 10
    };

    try {
      const res = await fetch('/api/draft', {
        method: 'POST', headers: {'Content-Type': 'application/json'},
        body: JSON.stringify(payload)
      }).then(r => r.json());

      if (res.ok) {
        currentDraft = { ...payload, ...res };
        const ve = res.vitals_eval;
        const dr = res.draft;
        const banner = document.getElementById('draftBanner');
        banner.className = `draft-banner tier-${ve.tier}`;
        document.getElementById('draftTierTitle').textContent = uiLang === 'en'
          ? `AI Preliminary Draft — ${ve.tier} TIER`
          : `مسودة اقتراح أولي — الفئة ${ve.tier}`;
        document.getElementById('draftScoreBadge').textContent = uiLang === 'en'
          ? `Severity Score: ${ve.score} / 10`
          : `درجة الخطورة: ${ve.score} / 10`;
        document.getElementById('draftMetaLine').textContent = uiLang === 'en'
          ? `Patient ID: ${res.patient_id} | Patient: ${payload.patient_name} (${payload.age ?? '?'}y, ${payload.gender}) | MAP: ${ve.map ?? '--'} mmHg | Shock Index: ${ve.shock_index ?? '--'}`
          : `رقم المريض: ${res.patient_id} | المريض: ${payload.patient_name} (${payload.age ?? '?'} سنة، ${payload.gender}) | MAP: ${ve.map ?? '--'} | مؤشر الصدمة: ${ve.shock_index ?? '--'}`;

        const ul = document.getElementById('draftAlertsList');
        ul.innerHTML = ve.alerts.map(a => `<li>${a}</li>`).join('');

        document.getElementById('draftModelBadge').textContent = `🤖 ${res.model}`;
        document.getElementById('rSummary').value = dr.clinical_summary;
        document.getElementById('rTier').value = ve.tier;
        document.getElementById('rScore').value = ve.score;
        document.getElementById('rDept').value = dr.suggested_department;
        document.getElementById('hitlConfirmCheck').checked = false;

        const cats = I18N[uiLang].prepCats;
        const grid = document.getElementById('prepGrid');
        grid.innerHTML = Object.keys(cats).map(k => {
          const items = (dr[k] || []).join(' • ');
          return `<div class="prep-card"><strong>${cats[k]}</strong><div style="margin-top:4px; color:#334155;">${items || '-'}</div></div>`;
        }).join('');

        document.getElementById('draftContainer').classList.remove('hidden');
        document.getElementById('draftContainer').scrollIntoView({ behavior: 'smooth' });
      }
    } finally {
      btn.disabled = false;
      btn.textContent = I18N[uiLang].genBtn;
    }
  }

  async function confirmAndSend() {
    if (!document.getElementById('hitlConfirmCheck').checked) {
      alert(uiLang === 'en'
        ? 'Please check the clinical accountability confirmation box before dispatching.'
        : 'يرجى تحديد مربع تأكيد المسؤولية السريرية قبل الإرسال.');
      return;
    }
    const btn = document.getElementById('confirmSendBtn');
    btn.disabled = true;

    const record = {
      lang: uiLang,
      patient_id: currentDraft.patient_id,
      patient_name: currentDraft.patient_name,
      age: currentDraft.age,
      gender: currentDraft.gender,
      national_id: currentDraft.national_id,
      sbp: currentDraft.sbp,
      dbp: currentDraft.dbp,
      map: currentDraft.vitals_eval.map,
      shock_index: currentDraft.vitals_eval.shock_index,
      hr: currentDraft.hr,
      spo2: currentDraft.spo2,
      rr: currentDraft.rr,
      temp: currentDraft.temp,
      avpu: currentDraft.avpu,
      gcs: currentDraft.gcs,
      medications: currentDraft.medications,
      eta: currentDraft.eta,
      tier: document.getElementById('rTier').value,
      score: Number(document.getElementById('rScore').value),
      department: document.getElementById('rDept').value,
      summary: document.getElementById('rSummary').value,
      prep: currentDraft.draft,
      alerts: currentDraft.vitals_eval.alerts,
      manually_edited: anyManualEdit,
      paramedic: 'EMS001',
      model: currentDraft.model
    };

    const res = await fetch('/api/confirm', {
      method: 'POST', headers: {'Content-Type': 'application/json'},
      body: JSON.stringify(record)
    }).then(r => r.json());

    btn.disabled = false;
    if (res.ok) {
      document.getElementById('draftContainer').classList.add('hidden');
      await refreshCases();
      alert(uiLang === 'en'
        ? `✅ Confirmed & Dispatched! Patient ID ${record.patient_id} is now live on the ER Dashboard.`
        : `✅ تم التأكيد والإرسال! الحالة ${record.patient_id} معروضة الآن في شاشة الطوارئ.`);
    }
  }

  function playChime() {
    if (!chimeEnabled) return;
    try {
      const ctx = new (window.AudioContext || window.webkitAudioContext)();
      const osc = ctx.createOscillator();
      osc.type = 'sine';
      osc.frequency.setValueAtTime(880, ctx.currentTime);
      osc.connect(ctx.destination);
      osc.start();
      osc.stop(ctx.currentTime + 0.25);
    } catch (_) {}
  }

  function toggleChime() {
    chimeEnabled = !chimeEnabled;
    applyLanguage();
  }

  async function refreshCases() {
    try {
      const cases = await fetch('/api/cases').then(r => r.json());
      let isNew = false;
      cases.forEach(c => {
        if (!knownCaseIds.has(c.patient_id)) {
          if (knownCaseIds.size > 0) isNew = true;
          knownCaseIds.add(c.patient_id);
        }
      });
      if (isNew) playChime();

      // تحديث جدول التدقيق في واجهة المسعف
      const tbody = document.getElementById('auditTableBody');
      tbody.innerHTML = cases.map(c => `
        <tr>
          <td><code>${c.patient_id}</code></td>
          <td><b>${c.tier}</b> (${c.score}/10)</td>
          <td>${c.patient_name} (${c.age ?? '?'}y, ${c.gender})</td>
          <td><code>${c.masked_nid}</code></td>
          <td>${c.sbp ?? '-'}/${c.dbp ?? '-'} (MAP ${c.map ?? '-'}) | HR ${c.hr ?? '-'}</td>
          <td>${c.eta}m</td>
          <td>${c.manually_edited ? '✎ Yes' : 'No'}</td>
        </tr>
      `).join('');

      // تحديث لوحة طوارئ المستشفى /er
      document.getElementById('cntTotal').textContent = cases.length;
      document.getElementById('cntRed').textContent = cases.filter(c => c.tier === 'RED').length;
      document.getElementById('cntOrange').textContent = cases.filter(c => c.tier === 'ORANGE').length;
      document.getElementById('cntGreen').textContent = cases.filter(c => c.tier === 'GREEN').length;

      const list = document.getElementById('erCasesList');
      if (!cases.length) {
        list.innerHTML = `<div class="card" style="text-align:center; padding:34px; color:#64748b;">${I18N[uiLang].noCases}</div>`;
        return;
      }

      const cats = I18N[uiLang].prepCats;
      list.innerHTML = cases.map(c => {
        const badgeBg = c.tier === 'RED' ? '#fee2e2; color:#991b1b' : (c.tier === 'ORANGE' ? '#ffedd5; color:#9a3412' : '#dcfce7; color:#166534');
        const prepHtml = Object.keys(cats).map(k => {
          const arr = (c.prep && c.prep[k]) || [];
          return arr.length ? `<div style="font-size:12.5px; margin-top:4px;"><b>${cats[k]}:</b> ${arr.join(' • ')}</div>` : '';
        }).join('');

        return `
          <div class="er-card ${c.tier}">
            <div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:8px;">
              <div>
                <span class="badge" style="background:${badgeBg};">${c.tier} — ${c.score}/10</span>
                <code style="margin-inline-start:8px; font-weight:700; color:#0284c7;">${c.patient_id}</code>
              </div>
              <span class="badge" style="background:#0f172a; color:#fff;">⏱️ ETA: ${c.eta} min</span>
            </div>
            <div style="margin-top:10px; font-size:14px;">
              <b>${c.patient_name}</b> | ${c.age ?? '?'} yrs | ${c.gender} |
              <span>National ID: <code>${c.masked_nid}</code></span> |
              <span>Dept: <b>${c.department}</b></span>
            </div>
            <div style="margin-top:8px; background:#f8fafc; padding:8px 12px; border-radius:8px; font-size:13px;">
              <b>BP:</b> ${c.sbp ?? '-'}/${c.dbp ?? '-'} mmHg (<b>MAP:</b> ${c.map ?? '-'} mmHg) |
              <b>HR:</b> ${c.hr ?? '-'} bpm | <b>SpO₂:</b> ${c.spo2 ?? '-'}% |
              <b>AVPU:</b> ${c.avpu || '-'} | <b>GCS:</b> ${c.gcs ?? '-'}
            </div>
            <div style="margin-top:8px; font-size:13.5px;"><b>Summary:</b> ${c.summary}</div>
            ${c.medications ? `<div style="margin-top:6px; background:#fef2f2; color:#991b1b; padding:6px 10px; border-radius:6px; font-size:12.5px;">💊 <b>Medications / Interaction Alert:</b> ${c.medications}</div>` : ''}
            <div style="margin-top:8px; border-top:1px solid #e2e8f0; padding-top:8px;">${prepHtml}</div>
          </div>
        `;
      }).join('');
    } catch (_) {}
  }

  setInterval(refreshCases, 3000);
  if (currentView === 'er') toggleView();
  else applyLanguage();
</script>
</body>
</html>
"""

class RelayRequestHandler(BaseHTTPRequestHandler):
    def log_message(self, format, *args):
        return

    def _send_json(self, obj: Any, status: int = 200):
        raw = json.dumps(obj, ensure_ascii=False).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(raw)))
        self.end_headers()
        self.wfile.write(raw)

    def do_GET(self):
        path = urlparse(self.path).path
        if path in ("/", "/er"):
            raw = HTML_PAGE.encode("utf-8")
            self.send_response(200)
            self.send_header("Content-Type", "text/html; charset=utf-8")
            self.send_header("Content-Length", str(len(raw)))
            self.end_headers()
            self.wfile.write(raw)
        elif path == "/api/cases":
            with STORE_LOCK:
                self._send_json(list(CASES_STORE))
        else:
            self._send_json({"error": "Not found"}, 404)

    def do_POST(self):
        path = urlparse(self.path).path
        length = int(self.headers.get("Content-Length", 0))
        body = json.loads(self.rfile.read(length).decode("utf-8") or "{}")
        try:
            if path == "/api/extract":
                self._send_json(api_extract_fields(body.get("text", ""), body.get("lang", "en"), body.get("mode", "deep")))
            elif path == "/api/transcribe_audio":
                self._send_json(api_transcribe_audio(body.get("audio_b64", ""), body.get("mime_type", "audio/webm"), body.get("lang", "en")))
            elif path == "/api/draft":
                self._send_json(api_generate_draft(body))
            elif path == "/api/confirm":
                self._send_json(api_confirm_case(body))
            else:
                self._send_json({"ok": False, "error": "Unknown endpoint"}, 404)
        except Exception as e:
            self._send_json({"ok": False, "error": str(e)}, 500)

def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

PORT = find_free_port()
server = ThreadingHTTPServer(("0.0.0.0", PORT), RelayRequestHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()

print(f"✅ RELAY Server running on port {PORT}")
try:
    from google.colab import output
    print("🌐 Click the link below to open RELAY in a full browser tab (Recommended for Live Microphone):")
    output.serve_kernel_port_as_window(PORT, path="/", anchor_text="🚀 Open RELAY Paramedic Terminal (Full Tab)")
    output.serve_kernel_port_as_window(PORT, path="/er", anchor_text="🏥 Open Hospital ER Live Dashboard (/er)")
except Exception:
    print(f"Open http://127.0.0.1:{PORT}/ and http://127.0.0.1:{PORT}/er")

✅ RELAY Server running on port 49507
🌐 Click the link below to open RELAY in a full browser tab (Recommended for Live Microphone):
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>